In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

In [ ]:
torch.manual_seed(42)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" using device {device}")

 using device cuda


In [ ]:
import pandas as pd
import os


dataset_path = "/content"

# Read the CSV files
df_train = pd.read_csv(os.path.join(dataset_path, "fashion-mnist_train.csv"))
df_test = pd.read_csv(os.path.join(dataset_path, "fashion-mnist_test.csv"))

# Check the data
print("Training set shape:", df_train.shape)
print("Test set shape:", df_test.shape)
print("\nFirst 5 rows of training data:")
df_train.head()

Training set shape: (60000, 785)
Test set shape: (10000, 785)

First 5 rows of training data:


,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
df_train.shape

(60000, 785)

In [ ]:
x = df_train.iloc[:, 1:].values
y = df_train.iloc[:, 0].values

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x , y , test_size= 0.2, random_state=42)

In [ ]:
#Transformations
from torchvision.transforms import transforms
custom_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

In [ ]:
from PIL import Image
class CustomDataset(Dataset):

    def __init__(self, features, labels, transform):
        self.features = features
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):

        #resize
        image = self.features[index].reshape(28,28)

        #datatype
        image  = image.astype(np.uint8)

        #blacknwhite to color
        image = np.stack([image]*3, axis= -1)

        #Convert into PIL
        image = Image.fromarray(image)

        #Apply transformations
        image = self.transform(image)

        return image, torch.tensor(self.labels[index], dtype=torch.long)

In [ ]:
train_dataset = CustomDataset(x_train, y_train, transform = custom_transform)

In [ ]:
test_dataset = CustomDataset(x_test, y_test, transform=custom_transform)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)

In [ ]:
#Fetch the pretrained model
import torchvision.models as models

vgg16 = models.vgg16(pretrained=True)



/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:06<00:00, 83.5MB/s]


In [ ]:
vgg16

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [ ]:
for param in vgg16.features.parameters():
    param.requires_grad=False

In [ ]:
vgg16.classifier= nn.Sequential(
    nn.Linear(25088, 1024),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Dropout(0.5),

    nn.Linear(512, 10)
)

In [ ]:
vgg16 = vgg16.to(device)

In [ ]:
learning_rate = 0.0001
epochs = 15

In [ ]:
criterion = nn.CrossEntropyLoss()
optomizer = optim.Adam(vgg16.classifier.parameters(), lr=learning_rate) #, weight_decay=1e-4

In [ ]:
for epoch in range(epochs):

    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:

        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

        # Forward pass
        y_pred = vgg16(batch_features)

        # Compute loss
        loss = criterion(y_pred,batch_labels)

        # Zero out previous gradients
        optomizer.zero_grad()

        # Back Propagation
        loss.backward()

        # Update weights
        optomizer.step()

        total_epoch_loss = total_epoch_loss + loss.item()

    avg_loss = total_epoch_loss / len(train_loader)
    print(f"Epoch: {epoch},  loss: {avg_loss}")

Epoch: 0,  loss: 0.3639915325169762
Epoch: 1,  loss: 0.212959860800455
Epoch: 2,  loss: 0.16463447467423975
Epoch: 3,  loss: 0.12645484505655866
Epoch: 4,  loss: 0.09826864007092082
Epoch: 5,  loss: 0.07826882499448645
Epoch: 6,  loss: 0.06398188719721899
Epoch: 7,  loss: 0.05282075194568218
Epoch: 8,  loss: 0.043058040046467794
Epoch: 9,  loss: 0.040389566387581
Epoch: 10,  loss: 0.034308551301728586
Epoch: 11,  loss: 0.0317557599296609
Epoch: 12,  loss: 0.028251234158350902
Epoch: 13,  loss: 0.0259122688471136
Epoch: 14,  loss: 0.025975675309699606


In [ ]:
vgg16.eval()

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [ ]:
total = 0
correct = 0

with torch.no_grad():
    for batch_features,batch_labels in test_loader:

        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

        y_pred = vgg16(batch_features)      # Forward pass

        _, predicted  = torch.max(y_pred, 1)    # Get predicted labels
        total = total + batch_labels.shape[0]
        correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.927


In [ ]:
# Save the entire model (simpler for prediction)
torch.save(vgg16, 'vgg16_fashion_mnist_full.pth')
print("Model saved as 'vgg16_fashion_mnist_full.pth'")

Model saved as 'vgg16_fashion_mnist_full.pth'
